# Cross-dataset view

Tail metrics for the four third-party datasets side by side. Run from `tertiary-repos/`.

| Cell | Output |
|---|---|
| 1 | `aggregated_tail_factor_heatmap.pdf`: TF per platform for SPM, NSD, BEF |
| 2 | `aggregated_ratio_curves.pdf`: all-calls / warm-only ratio, p80-p100 |
| 3 | `nsd_ratio_curves_by_size_and_func_type.pdf` |
| 4 | `intraday_tail_factors_spm_nsd_huawei.pdf` |
| 5 | NSD intraday table |
| 6 | `config_tail_factor_heatmap.pdf`: TF per Huawei configuration |

HUA is read from `../data/tertiary_aggregations/*.csv`, so the raw Huawei trace is not needed.

In [1]:
# Aggregated tail-factor heatmap across SPM / NSD / BEF (independent cell)
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import duckdb
from sqlalchemy import create_engine

# --- Config (paths relative to tertiary-repos/, where this notebook lives) ---
PERCENTILES = [95, 99, 99.9, 99.99]
COL_LABELS = [fr"$TF_{{p{p}}}$" for p in PERCENTILES]

SPM_DB = Path("mahmoudi_et_al/experiments/results/results.duckdb")
SPM_TABLE = "results"
SPM_DURATION_COL = "client_elapsed_time"
SPM_ELAPSED_COL = "elapsed_time"
SPM_IS_COLD_COL = "is_cold"

BEF_DB = Path("kuhlenkamp_et_al/full_data.duckdb")
BEF_TABLE = "full_data_enriched"
BEF_PROVIDER_COL = "Provider"
BEF_DLAT_COL = "DLat"

# NSD is read straight from the local MySQL import (see README)
NSD_MYSQL_URL = "mysql+pymysql://root:@localhost/benchmark"
NSD_TABLE = "data"
NSD_INVOKE_COL = "invoke_time"
NSD_RETURN_COL = "return_time"
NSD_TYPE_COL = "type"
NSD_ISCOLD_COL = "isCold"

# Paper-friendly global font update
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
})


def get_short_year_label(dataset: str, raw_platform: str) -> tuple[str, str, str]:
    """
    Returns (group_key, plot_latex_label, terminal_print_label)
    Schema: $platformShort$Year (e.g. Lambda20, GCF23).
    Handles Alpha/Beta collisions for Lambda in 2020.
    """
    p_lower = raw_platform.lower()
    
    # Extract architecture suffixes if present
    suffix = ""
    if "x86" in p_lower: suffix = " (X86)"
    elif "arm" in p_lower: suffix = " (ARM)"
    
    # Standardize base platform names and establish grouping keys
    if "aws" in p_lower or "lambda" in p_lower:
        base = "Lambda"; g_key = "Lambda"
    elif "gcf" in p_lower or "google" in p_lower:
        base = "GCF"; g_key = "GCF"
    elif "azure" in p_lower:
        base = "Azure"; g_key = "Azure"
    elif "ibm" in p_lower:
        base = "IBM"; g_key = "IBM"
    elif "oracle" in p_lower or "ocf" in p_lower:
        base = "OCF"; g_key = "OCF"
    elif "cloudflare" in p_lower or "cw" in p_lower:
        base = "CW"; g_key = "CW"
    elif "deno" in p_lower or "dd" in p_lower:
        base = "DD"; g_key = "DD"
    elif "fastly" in p_lower:
        base = "Fastly"; g_key = "Fastly"
    elif "flyio" in p_lower or "fly.io" in p_lower:
        base = "Fly.io"; g_key = "Fly.io"
    else:
        base = raw_platform.capitalize(); g_key = base

    # Apply year and alpha/beta suffixes
    if dataset == "SPM": # 2020
        if base == "Lambda":
            return g_key, r"Lambda20$^{\alpha}$" + suffix, "Lambda20(a)" + suffix
        return g_key, f"{base}20{suffix}", f"{base}20{suffix}"
        
    elif dataset == "BEF": # 2020
        if base == "Lambda":
            return g_key, r"Lambda20$^{\beta}$" + suffix, "Lambda20(b)" + suffix
        return g_key, f"{base}20{suffix}", f"{base}20{suffix}"
        
    elif dataset == "NSD": # 2023
        return g_key, f"{base}23{suffix}", f"{base}23{suffix}"
    
    return g_key, base + suffix, base + suffix


def _as_warm_bool(series: pd.Series) -> pd.Series:
    """Normalize common boolean encodings and return a warm-mask (True means warm)."""
    s = series.astype(str).str.strip().str.lower()
    return s.isin(["false", "0", "no"])

def _to_datetime_auto(series: pd.Series) -> pd.Series:
    """Automatically converts raw timestamps to datetimes by inferring the unit."""
    if pd.api.types.is_datetime64_any_dtype(series):
        return series
    numeric = pd.to_numeric(series, errors="coerce")
    if float(numeric.notna().mean()) >= 0.9:
        med = float(np.nanmedian(np.abs(numeric.to_numpy(dtype=float))))
        if med > 1e17: unit = "ns"
        elif med > 1e14: unit = "us"
        elif med > 1e11: unit = "ms"
        elif med > 1e8: unit = "s"
        else: unit = "ms"
        return pd.to_datetime(numeric, unit=unit, errors="coerce")
    return pd.to_datetime(series, errors="coerce")


def compute_tail_factors(values: np.ndarray, percentiles: list[float]) -> tuple[np.ndarray, float, np.ndarray]:
    """Computes Tail Factor (q / median), the baseline median, and the raw percentile values."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError("No finite values to compute tail factors.")
    median = float(np.median(values))
    if not np.isfinite(median) or median <= 0:
        raise ValueError(f"Invalid median: {median}")
    q = np.percentile(values, percentiles)
    factors = q / median
    return factors.astype(float), median, q.astype(float)


def load_spm_row() -> tuple[str, np.ndarray, float, np.ndarray]:
    if not SPM_DB.exists():
        raise FileNotFoundError(f"SPM DuckDB not found: {SPM_DB.resolve()}")
    con = duckdb.connect(str(SPM_DB))
    try:
        df = con.execute(
            f"SELECT {SPM_DURATION_COL} AS client_elapsed_s, {SPM_ELAPSED_COL} AS elapsed_ms, {SPM_IS_COLD_COL} AS is_cold FROM {SPM_TABLE}"
        ).df()
    finally:
        con.close()

    df["client_elapsed_s"] = pd.to_numeric(df["client_elapsed_s"], errors="coerce")
    df["elapsed_ms"] = pd.to_numeric(df["elapsed_ms"], errors="coerce")
    df["duration_ms"] = df["client_elapsed_s"] * 1000.0 - df["elapsed_ms"]
    df = df.dropna(subset=["duration_ms"])

    warm_mask = _as_warm_bool(df["is_cold"])
    warm = df.loc[warm_mask, "duration_ms"].to_numpy()
    factors, median, absolute_quantiles = compute_tail_factors(warm, PERCENTILES)
    return "aws", factors, median, absolute_quantiles


def load_nsd_row() -> tuple[str, np.ndarray, float, np.ndarray]:
    engine = create_engine(NSD_MYSQL_URL)
    df = pd.read_sql(
        f"SELECT google_time, function_start, function_stop, {NSD_TYPE_COL}, {NSD_ISCOLD_COL} FROM {NSD_TABLE}",
        engine,
    )

    # 1. Isolate warm starts first (isCold == False only)
    warm_mask = _as_warm_bool(df[NSD_ISCOLD_COL])
    df = df.loc[warm_mask].copy()

    # 2. Calculate worker runtime
    start_dt = _to_datetime_auto(df["function_start"])
    stop_dt = _to_datetime_auto(df["function_stop"])
    runtime_ms = (stop_dt - start_dt).dt.total_seconds() * 1000.0

    # 3. Unit normalization heuristic for google_time
    google_raw = pd.to_numeric(df["google_time"], errors="coerce")
    google_ms = google_raw.copy()
    
    try:
        runtime_med = float(np.nanmedian(runtime_ms.to_numpy(dtype=float)))
        google_med = float(np.nanmedian(google_raw.to_numpy(dtype=float)))
        if np.isfinite(runtime_med) and runtime_med > 0 and np.isfinite(google_med) and google_med > 0:
            if (google_med / runtime_med) < 0.05:
                google_ms = google_raw * 1000.0
    except Exception:
        pass

    # 4. Calculate Platform Latency
    df["duration_ms"] = google_ms - runtime_ms
    
    # 5. Filter out invalid rows and negative latencies (clock drift)
    df = df.dropna(subset=["duration_ms"])
    df = df[df["duration_ms"] >= 0]

    warm = df["duration_ms"].to_numpy()
    if warm.size == 0:
        raise ValueError("NSD warm selection produced 0 rows.")
        
    factors, median, absolute_quantiles = compute_tail_factors(warm, PERCENTILES)
    return "gcf", factors, median, absolute_quantiles


def load_bef_rows() -> tuple[list[str], np.ndarray, list[float], np.ndarray]:
    if not BEF_DB.exists():
        raise FileNotFoundError(f"BEF DuckDB not found: {BEF_DB.resolve()}")
    con = duckdb.connect(str(BEF_DB))
    try:
        df = con.execute(
            f"""
            SELECT {BEF_PROVIDER_COL} AS provider, {BEF_DLAT_COL} AS dlat
            FROM {BEF_TABLE}
            WHERE RSuccessed = TRUE AND CNew = 'Reused' AND {BEF_DLAT_COL} IS NOT NULL
            """
        ).df()
    finally:
        con.close()

    df["dlat"] = pd.to_numeric(df["dlat"], errors="coerce")
    df = df[np.isfinite(df["dlat"])].copy()
    if df.empty:
        raise ValueError("BEF query returned no finite DLat values.")

    rows, medians, quantiles, labels = [], [], [], []
    for provider in sorted(df["provider"].dropna().unique()):
        vals = df.loc[df["provider"] == provider, "dlat"].to_numpy()
        if vals.size == 0:
            continue
        try:
            factors, median, absolute_quantiles = compute_tail_factors(vals, PERCENTILES)
        except ValueError:
            continue
        labels.append(str(provider))
        rows.append(factors)
        medians.append(median)
        quantiles.append(absolute_quantiles)

    if not rows:
        raise ValueError("BEF produced 0 provider rows.")
    return labels, np.vstack(rows), medians, np.vstack(quantiles)


def plot_grouped_heatmap(
    matrix: np.ndarray,
    row_labels: list[str],
    group_spans: list[tuple[str, int, int]],
    col_labels: list[str],
    colorbar_label: str,
    *,
    cmap: str = "viridis",
) -> None:
    nrows, ncols = matrix.shape

    # Standard 1-column ACM width is 3.33in. Calculate height proportionally.
    fig_w = 3.33
    fig_h = max(2.5, 0.25 * nrows)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=300)

    valid = np.isfinite(matrix) & (matrix > 0)
    if np.any(valid):
        vmin = float(np.nanmin(matrix[valid]))
        vmax = float(np.nanmax(matrix[valid]))
        norm = LogNorm(vmin=vmin, vmax=vmax)
    else:
        norm = None

    im = ax.imshow(matrix, aspect="auto", cmap=cmap, norm=norm)

    # Cbar formatting
    cbar = fig.colorbar(im, ax=ax, pad=0.04, fraction=0.05)
    cbar.set_label(colorbar_label, fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    cbar.ax.minorticks_off() 

    ax.set_xticks(range(ncols))
    ax.set_xticklabels(col_labels, fontsize=7)
    ax.set_yticks(range(nrows))
    ax.set_yticklabels(row_labels, fontsize=7)

    # Tick params
    ax.tick_params(axis="y", pad=4)
    ax.tick_params(axis="x", pad=4)
    ax.set_xlabel("Percentile", fontsize=8)

    # Visual grouping: horizontal separators only
    for name, start, end in group_spans:
        if start > 0:
            ax.axhline(start - 0.5, color="white", linewidth=1.2, alpha=0.9)

    # Annotate values with dynamic contrast
    if nrows <= 30 and norm is not None:
        for i in range(nrows):
            for j in range(ncols):
                v = matrix[i, j]
                if np.isfinite(v):
                    norm_val = norm(v)
                    text_color = "black" if norm_val > 0.55 else "white"
                    ax.text(j, i, f"{v:.1f}", ha="center", va="center", color=text_color, fontsize=7)

    fig.subplots_adjust(left=0.36, right=0.88, bottom=0.15, top=0.95)

    plt.savefig("../figures/tertiary/aggregated_tail_factor_heatmap.pdf", bbox_inches="tight")
    plt.show()


print("Data sources used by aggregated plot:")
print(f"- SPM DuckDB: {SPM_DB} (table: {SPM_TABLE})")
print(f"- NSD MySQL: {NSD_MYSQL_URL} (table: {NSD_TABLE})")
print(f"- BEF DuckDB: {BEF_DB} (table: {BEF_TABLE})")

spm_label, spm_row, spm_median, spm_quantiles = load_spm_row()
nsd_label, nsd_row, nsd_median, nsd_quantiles = load_nsd_row()
bef_labels, bef_rows, bef_medians, bef_quantiles = load_bef_rows()

# --- Group rows by platform, containing absolute latency tracking metrics ---
# Format: (group_key, plot_latex_label, terminal_print_label, factors, median, absolute_qs)
rows = []  

g_key, p_plot, p_print = get_short_year_label("SPM", spm_label)
rows.append((g_key, p_plot, p_print, spm_row, spm_median, spm_quantiles))

g_key, p_plot, p_print = get_short_year_label("NSD", nsd_label)
rows.append((g_key, p_plot, p_print, nsd_row, nsd_median, nsd_quantiles))

for lbl, factors, med, absolute_qs in zip(bef_labels, bef_rows, bef_medians, bef_quantiles):
    g_key, p_plot, p_print = get_short_year_label("BEF", lbl)
    rows.append((g_key, p_plot, p_print, factors, med, absolute_qs))

PREFERRED_PROVIDER_ORDER = ["Lambda", "GCF", "Azure", "IBM", "OCF"]
provider_order = {p: i for i, p in enumerate(PREFERRED_PROVIDER_ORDER)}

rows_sorted = sorted(
    rows,
    key=lambda r: (
        provider_order.get(r[0], 999), # Sort by preferred platform blocks
        r[2]                           # Alphabetical by terminal label to cluster years/archs cleanly
    ),
)

row_labels = [r[1] for r in rows_sorted]
matrix = np.vstack([r[3].reshape(1, -1) for r in rows_sorted])

# Determine where to draw the white horizontal divider lines (by provider group)
group_spans: list[tuple[str, int, int]] = []
current = None
start = 0
for idx, row in enumerate(rows_sorted):
    prov = row[0]
    if current is None:
        current = prov
        start = 0
    elif prov != current:
        group_spans.append((current, start, idx))
        current = prov
        start = idx
if current is not None:
    group_spans.append((current, start, len(rows_sorted)))

# Draw the visual heatmap representation
plot_grouped_heatmap(
    matrix,
    row_labels,
    group_spans,
    COL_LABELS,
    colorbar_label="Tail Factor (TF)",
)

# --- Construct and display absolute latency tables ---
absolute_table_rows = []
for row in rows_sorted:
    absolute_table_rows.append({
        "Platform": row[2],
        "Median": row[4],
        "p95": row[5][0],
        "p99": row[5][1],
        "p99.9": row[5][2],
        "p99.99": row[5][3]
    })

df_absolute_latencies = pd.DataFrame(absolute_table_rows)

print("\n" + "="*85)
print("Absolute Latency Values per Percentile (Units matching native dataset metrics, e.g. ms)")
print("="*85)
print(df_absolute_latencies.round(2).to_string(index=False))
print("="*85 + "\n")

Data sources used by aggregated plot:
- SPM DuckDB: mahmoudi_et_al/experiments/results/results.duckdb (table: results)
- NSD MySQL: mysql+pymysql://root:@localhost/benchmark (table: data)
- BEF DuckDB: kuhlenkamp_et_al/full_data.duckdb (table: full_data_enriched)



Absolute Latency Values per Percentile (Units matching native dataset metrics, e.g. ms)
   Platform  Median    p95    p99  p99.9  p99.99
Lambda20(a)   92.43 174.98 351.03 560.02 1703.47
Lambda20(b)    0.03   0.06   0.09  13.89   14.56
      GCF20    0.08   8.71  18.55  21.95   27.64
      GCF23   10.27 100.47 121.85 202.07  296.68
    Azure20   18.82  27.80  29.02  29.54   29.70
      IBM20    3.51   6.24   8.10  11.59   19.70



<notebook cell>:299: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Aggregated percentile ratio curves (p80–p100): all / warm
from __future__ import annotations

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from matplotlib.ticker import FuncFormatter
import duckdb
from sqlalchemy import create_engine

# --- Paper-friendly styling (ACM 1-Column) ---
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'lines.linewidth': 1.2,
})

PERCENTILE_GRID = np.linspace(0, 100, 1000)
TARGET_PERCENTILES = [80, 95, 99, 99.9, 99.99]

SPM_DB = Path("mahmoudi_et_al/experiments/results/results.duckdb")
SPM_TABLE = "results"
SPM_DURATION_COL = "client_elapsed_time"
SPM_ELAPSED_COL = "elapsed_time"
SPM_IS_COLD_COL = "is_cold"

BEF_DB = Path("kuhlenkamp_et_al/full_data.duckdb")
BEF_TABLE = "full_data_enriched"
BEF_PROVIDER_COL = "Provider"
BEF_DLAT_COL = "DLat"

NSD_MYSQL_URL = "mysql+pymysql://root:@localhost/benchmark"
NSD_TABLE = "data"
NSD_ISCOLD_COL = "isCold"

HUAWEI_CSV_PATH = Path("../data/tertiary_aggregations/reconstruct_coldstart_platform_impact_percentiles.csv")

LINE_STYLES = {"SPM": "--", "NSD": ":", "BEF": "-", "HUAWEI": "-."}
PLATFORM_COLORS = {
    "lambda": "#1f77b4", 
    "gcf": "#2ca02c", 
    "azure": "#ff7f0e", 
    "ibm": "#9467bd",
    "huawei": "#d62728"
}

def get_labels(dataset: str, platform_raw: str) -> tuple[str, str, str]:
    """
    Returns (short_platform_name, plot_latex_label, terminal_print_label)
    Schema: $platformShort$Year (e.g. Lambda20, GCF23).
    Handles Alpha/Beta collisions for Lambda in 2020.
    """
    p_lower = platform_raw.strip().lower()
    if "aws" in p_lower or "lambda" in p_lower:
        p_short = "Lambda"
    elif "gcf" in p_lower or "google" in p_lower:
        p_short = "GCF"
    elif "azure" in p_lower:
        p_short = "Azure"
    elif "ibm" in p_lower:
        p_short = "IBM"
    elif "huawei" in p_lower:
        p_short = "Huawei"
    else:
        p_short = platform_raw.capitalize()
        
    if dataset == "SPM": # 2020
        if p_short == "Lambda":
            return p_short, r"Lambda20$^{\alpha}$", "Lambda20(a)"
        return p_short, f"{p_short}20", f"{p_short}20"
        
    elif dataset == "BEF": # 2020
        if p_short == "Lambda":
            return p_short, r"Lambda20$^{\beta}$", "Lambda20(b)"
        return p_short, f"{p_short}20", f"{p_short}20"
        
    elif dataset == "NSD": # 2023
        return p_short, f"{p_short}23", f"{p_short}23"
        
    elif dataset == "HUAWEI": # 2023
        return p_short, f"{p_short}25", f"{p_short}25"
    
    return p_short, p_short, p_short

def _as_warm_bool(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    return s.isin(["false", "0", "no"])

def _ratio_curve(all_vals: np.ndarray, warm_vals: np.ndarray, percentiles: list | np.ndarray) -> np.ndarray:
    q_all = np.percentile(all_vals, percentiles)
    q_warm = np.percentile(warm_vals, percentiles)
    return np.divide(q_all, q_warm, out=np.full_like(q_all, np.nan, dtype=float), where=q_warm != 0)

def _color_for_platform(p_short: str) -> str:
    n = p_short.lower()
    for key in PLATFORM_COLORS:
        if key in n:
            return PLATFORM_COLORS[key]
    return "#7f7f7f"

def load_spm_curve() -> tuple[str, np.ndarray, list[float]]:
    if not SPM_DB.exists():
        raise FileNotFoundError(f"SPM DuckDB not found: {SPM_DB.resolve()}")
    with duckdb.connect(str(SPM_DB)) as con:
        df = con.execute(
            f"SELECT {SPM_DURATION_COL} AS client_elapsed_s, {SPM_ELAPSED_COL} AS elapsed_ms, {SPM_IS_COLD_COL} AS is_cold FROM {SPM_TABLE}"
        ).df()

    df["client_elapsed_s"] = pd.to_numeric(df["client_elapsed_s"], errors="coerce")
    df["elapsed_ms"] = pd.to_numeric(df["elapsed_ms"], errors="coerce")
    df["duration_ms"] = df["client_elapsed_s"] * 1000.0 - df["elapsed_ms"]
    df = df.dropna(subset=["duration_ms"])
    warm_mask = _as_warm_bool(df["is_cold"])
    
    dur_all = df["duration_ms"].to_numpy()
    dur_warm = df.loc[warm_mask, "duration_ms"].to_numpy()
    if dur_warm.size == 0:
        raise ValueError("SPM warm selection produced 0 rows.")
    return "aws", _ratio_curve(dur_all, dur_warm, PERCENTILE_GRID), _ratio_curve(dur_all, dur_warm, TARGET_PERCENTILES).tolist()

def load_nsd_curve() -> tuple[str, np.ndarray, list[float]]:
    def _to_datetime_auto(series: pd.Series) -> pd.Series:
        if pd.api.types.is_datetime64_any_dtype(series):
            return series

        numeric = pd.to_numeric(series, errors="coerce")
        frac_numeric = float(numeric.notna().mean())

        if frac_numeric >= 0.9:
            med = float(np.nanmedian(np.abs(numeric.to_numpy(dtype=float))))
            if med > 1e17: unit = "ns"
            elif med > 1e14: unit = "us"
            elif med > 1e11: unit = "ms"
            elif med > 1e8: unit = "s"
            else: unit = "ms"
            return pd.to_datetime(numeric, unit=unit, errors="coerce")
        return pd.to_datetime(series, errors="coerce")

    engine = create_engine(NSD_MYSQL_URL)
    df = pd.read_sql(
        f"SELECT google_time, function_start, function_stop, {NSD_ISCOLD_COL} FROM {NSD_TABLE}",
        engine,
    )

    google_raw = pd.to_numeric(df["google_time"], errors="coerce")
    start_dt = _to_datetime_auto(df["function_start"])
    stop_dt = _to_datetime_auto(df["function_stop"])
    runtime_ms = (stop_dt - start_dt).dt.total_seconds() * 1000.0

    google_ms = google_raw.copy()
    try:
        runtime_med = float(np.nanmedian(runtime_ms.to_numpy(dtype=float)))
        google_med = float(np.nanmedian(google_raw.to_numpy(dtype=float)))
        if np.isfinite(runtime_med) and runtime_med > 0 and np.isfinite(google_med) and google_med > 0:
            if (google_med / runtime_med) < 0.05:
                google_ms = google_raw * 1000.0
    except Exception:
        pass

    df["duration"] = google_ms - runtime_ms
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["duration"])
    df = df[df["duration"] >= 0]

    is_cold_raw = df[NSD_ISCOLD_COL].astype(str).str.strip().str.lower()
    warm_mask = is_cold_raw == "false"

    dur_all = df["duration"].to_numpy()
    dur_warm = df.loc[warm_mask, "duration"].to_numpy()
    if dur_warm.size == 0:
        raise ValueError("NSD warm selection produced 0 rows.")
    return "gcf", _ratio_curve(dur_all, dur_warm, PERCENTILE_GRID), _ratio_curve(dur_all, dur_warm, TARGET_PERCENTILES).tolist()

def load_bef_curves() -> list[tuple[str, np.ndarray, list[float]]]:
    if not BEF_DB.exists():
        raise FileNotFoundError(f"BEF DuckDB not found: {BEF_DB.resolve()}")
    with duckdb.connect(str(BEF_DB)) as con:
        df_all = con.execute(
            f"""
            SELECT {BEF_PROVIDER_COL} AS provider, {BEF_DLAT_COL} AS dlat
            FROM {BEF_TABLE}
            WHERE {BEF_DLAT_COL} IS NOT NULL AND {BEF_PROVIDER_COL} IS NOT NULL AND RSuccessed = TRUE
            """
        ).df()
        df_warm = con.execute(
            f"""
            SELECT {BEF_PROVIDER_COL} AS provider, {BEF_DLAT_COL} AS dlat
            FROM {BEF_TABLE}
            WHERE {BEF_DLAT_COL} IS NOT NULL AND {BEF_PROVIDER_COL} IS NOT NULL AND CNew = 'Reused' AND RSuccessed = TRUE
            """
        ).df()

    df_all["dlat"] = pd.to_numeric(df_all["dlat"], errors="coerce")
    df_warm["dlat"] = pd.to_numeric(df_warm["dlat"], errors="coerce")
    df_all = df_all[np.isfinite(df_all["dlat"])].copy()
    df_warm = df_warm[np.isfinite(df_warm["dlat"])].copy()

    providers = sorted(set(df_all["provider"]).intersection(df_warm["provider"]))
    curves: list[tuple[str, np.ndarray, list[float]]] = []
    
    for provider in providers:
        all_vals = df_all.loc[df_all["provider"] == provider, "dlat"].to_numpy()
        warm_vals = df_warm.loc[df_warm["provider"] == provider, "dlat"].to_numpy()
        if all_vals.size == 0 or warm_vals.size == 0:
            continue
        curves.append((
            str(provider), 
            _ratio_curve(all_vals, warm_vals, PERCENTILE_GRID),
            _ratio_curve(all_vals, warm_vals, TARGET_PERCENTILES).tolist()
        ))
    return curves

def load_huawei_curve() -> tuple[np.ndarray, list[float]]:
    if not HUAWEI_CSV_PATH.exists():
        print(f"Warning: Huawei CSV not found at {HUAWEI_CSV_PATH}")
        return np.full_like(PERCENTILE_GRID, np.nan), [np.nan] * len(TARGET_PERCENTILES)
    
    df = pd.read_csv(HUAWEI_CSV_PATH).sort_values("percentile")
    ratio_interp_grid = np.interp(PERCENTILE_GRID, df["percentile"], df["ratio_all_to_warm"], left=np.nan, right=np.nan)
    ratio_interp_targets = np.interp(TARGET_PERCENTILES, df["percentile"], df["ratio_all_to_warm"], left=np.nan, right=np.nan).tolist()
    return ratio_interp_grid, ratio_interp_targets

print("Fetching and computing percentiles. This may take a moment...")
spm_name, spm_ratio, spm_table = load_spm_curve()
nsd_name, nsd_ratio, nsd_table = load_nsd_curve()
bef_curves = load_bef_curves()
huawei_ratio, huawei_table = load_huawei_curve()

curves_data = []

def add_curve(dataset: str, raw_name: str, ratio_arr: np.ndarray, table_arr: list[float], z_order: int):
    p_short, plot_lbl, print_lbl = get_labels(dataset, raw_name)
    curves_data.append({
        "dataset": dataset,
        "platform": p_short,
        "ratio": ratio_arr,
        "plot_label": plot_lbl,
        "print_label": print_lbl,
        "table": table_arr,
        "z": z_order
    })

add_curve("SPM", spm_name, spm_ratio, spm_table, 3)
add_curve("NSD", nsd_name, nsd_ratio, nsd_table, 3)
for provider, ratio, tbl in bef_curves:
    add_curve("BEF", provider, ratio, tbl, 2)
if not np.isnan(huawei_ratio).all():
    add_curve("HUAWEI", "Huawei Cloud", huawei_ratio, huawei_table, 4)

# Sort curves alphabetically by their print label so the legend and table look organized
curves_data.sort(key=lambda x: x["print_label"])

# --- Terminal Summary Table ---
table_rows = []
for item in curves_data:
    t = item["table"]
    table_rows.append({
        "Curve": item["print_label"], 
        "p80": t[0], "p95": t[1], "p99": t[2], "p99.9": t[3], "p99.99": t[4]
    })

df_table = pd.DataFrame(table_rows)
print("\n" + "="*80)
print("Cold Factor (CF) Summary for Target Percentiles (p80 - p99.99)")
print("="*80)
print(df_table.round(3).to_string(index=False))
print("="*80 + "\n")

# --- Plotting (ACM Form Factor) ---
fig, ax = plt.subplots(figsize=(3.33, 3.2), dpi=300)

for item in curves_data:
    ax.plot(
        PERCENTILE_GRID,
        item["ratio"],
        label=item["plot_label"],
        linestyle=LINE_STYLES.get(item["dataset"], "-"),
        color=_color_for_platform(item["platform"]),
        alpha=0.85,
        zorder=item["z"]
    )

ax.set_xlabel("Percentile (%)")
ax.set_ylabel("Cold Factor (CF)")
ax.set_xlim(80, 100)

ax.set_yscale("log", base=10)
ax.set_ylim(bottom=1, top=100)
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:g}'))
ax.minorticks_off() 
ax.grid(True, linestyle="--", alpha=0.3, linewidth=0.6, zorder=1)

# --- Legend Sorting (Row-Wise Transposition) ---
handles, labels = ax.get_legend_handles_labels()
ncol = 4
nrows = int(np.ceil(len(handles) / ncol))

# Pad the handles and labels with invisible items to make a perfect grid
pad_size = (nrows * ncol) - len(handles)
padded_handles = handles + [Line2D([], [], linestyle='none')] * pad_size
padded_labels = labels + [''] * pad_size

# Reorder indices for visually left-to-right reading in a column-major setup
reordered_handles = []
reordered_labels = []
for i in range(ncol):
    for j in range(nrows):
        idx = j * ncol + i
        reordered_handles.append(padded_handles[idx])
        reordered_labels.append(padded_labels[idx])

# Compact flat legend above the plot
ax.legend(
    reordered_handles,
    reordered_labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=ncol,
    frameon=False,
    handlelength=1.5,
    columnspacing=1.0,
    handletextpad=0.4
)

axins = inset_axes(
    ax,
    width="45%",
    height="35%",
    loc="upper left",
    borderpad=2.5,
)

for item in curves_data:
    axins.plot(
        PERCENTILE_GRID,
        item["ratio"],
        linestyle=LINE_STYLES.get(item["dataset"], "-"),
        color=_color_for_platform(item["platform"]),
        linewidth=1.0,
        alpha=0.9
    )

axins.set_xlim(97.5, 100)
axins.set_ylim(bottom=1, top=1.25) 
axins.minorticks_off()
axins.grid(True, linestyle="--", alpha=0.25, linewidth=0.5)

axins.set_xticks([97.5, 98.5, 100])
axins.set_xticklabels(
    ["97.5", "98.5", "100"], 
    rotation=35,
    ha="right",
    rotation_mode="anchor"
)

for label in axins.get_xticklabels():
    if label.get_text() == "100":
        label.set_ha("center")

axins.set_yticks([1, 1.1, 1.2])
axins.set_yticklabels(["1", "1.1", "1.2"])
axins.tick_params(axis='both', labelsize=6, pad=1)

mark_inset(
    ax, axins, 
    loc1=3, loc2=4,
    fc="gray", alpha=0.15,
    ec="gray", lw=0.8,
    zorder=0
)

plt.tight_layout(rect=[0, 0, 1, 0.92])
os.makedirs("../figures/tertiary", exist_ok=True)
plt.savefig("../figures/tertiary/aggregated_ratio_curves.pdf", bbox_inches="tight")
plt.show()

Fetching and computing percentiles. This may take a moment...



Cold Factor (CF) Summary for Target Percentiles (p80 - p99.99)
      Curve    p80    p95    p99  p99.9  p99.99
    Azure20  0.982  0.995  0.996  1.000   1.000
      GCF20  1.015  1.007  1.007  1.002   1.000
      GCF23 87.823 18.119 18.216 15.831  19.637
   Huawei25  1.001  1.003  1.030  1.029   1.014
      IBM20  1.007  1.033  1.101  1.144   1.041
Lambda20(a)  1.000  1.019  1.005  1.068   1.000
Lambda20(b)  1.000  1.100 39.915  1.007   1.026



<notebook cell>:380: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.92])
<notebook cell>:383: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# NSD-only: cold-factor curves by function size and by function *type*
# Uses platform-overhead duration: google_time - (function_stop - function_start)

required = ["PERCENTILE_GRID", "NSD_MYSQL_URL", "NSD_TABLE", "NSD_ISCOLD_COL"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Run Cell 3 first (missing: {missing}).")

engine = create_engine(NSD_MYSQL_URL)

# Pull NSD fields needed for per-group analysis
nsd = pd.read_sql(
    f"""
    SELECT
      google_time,
      function_start,
      function_stop,
      {NSD_ISCOLD_COL} AS is_cold,
      size,
      func_name
    FROM {NSD_TABLE}
    """,
    engine,
)

# --- Derive platform-overhead duration (ms) ---
def _to_datetime_auto(series: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(series):
        return series

    numeric = pd.to_numeric(series, errors="coerce")
    frac_numeric = float(numeric.notna().mean())

    if frac_numeric >= 0.9:
        med = float(np.nanmedian(np.abs(numeric.to_numpy(dtype=float))))
        if med > 1e17:
            unit = "ns"
        elif med > 1e14:
            unit = "us"
        elif med > 1e11:
            unit = "ms"
        elif med > 1e8:
            unit = "s"
        else:
            unit = "ms"
        return pd.to_datetime(numeric, unit=unit, errors="coerce")

    return pd.to_datetime(series, errors="coerce")

google_raw = pd.to_numeric(nsd["google_time"], errors="coerce")
start_dt = _to_datetime_auto(nsd["function_start"])
stop_dt = _to_datetime_auto(nsd["function_stop"])
runtime_ms = (stop_dt - start_dt).dt.total_seconds() * 1000.0

# If google_time appears to be in seconds, normalize to ms.
google_ms = google_raw.copy()
try:
    runtime_med = float(np.nanmedian(runtime_ms.to_numpy(dtype=float)))
    google_med = float(np.nanmedian(google_raw.to_numpy(dtype=float)))
    if np.isfinite(runtime_med) and runtime_med > 0 and np.isfinite(google_med) and google_med > 0:
        if (google_med / runtime_med) < 0.05:
            google_ms = google_raw * 1000.0
except Exception:
    pass

nsd["duration"] = google_ms - runtime_ms
nsd = nsd.replace([np.inf, -np.inf], np.nan).dropna(subset=["duration", "is_cold", "size", "func_name"])
nsd = nsd[nsd["duration"] >= 0]

# Warm mask
is_cold_raw = nsd["is_cold"].astype(str).str.strip().str.lower()
nsd["is_warm"] = is_cold_raw.isin(["false", "0", "no"])

# Function type = prefix before first underscore (e.g., foo_12 -> foo)
nsd["func_type"] = nsd["func_name"].astype(str).str.strip().str.split("_", n=1).str[0]

# --- Build ratio curves per group ---
def _ratio_curve_for_group(df_g: pd.DataFrame) -> np.ndarray | None:
    all_vals = df_g["duration"].to_numpy(dtype=float)
    warm_vals = df_g.loc[df_g["is_warm"], "duration"].to_numpy(dtype=float)
    if warm_vals.size == 0 or all_vals.size == 0:
        return None
    # Cells 2/3 redefine _ratio_curve with an explicit percentile argument; pass the grid this
    # cell already requires so the notebook also runs top-to-bottom, not just out of order.
    return _ratio_curve(all_vals, warm_vals, PERCENTILE_GRID)

curves_by_size: dict[str, np.ndarray] = {}
for size, df_g in nsd.groupby("size"):
    ratio = _ratio_curve_for_group(df_g)
    if ratio is None:
        continue
    curves_by_size[str(size)] = ratio

curves_by_type: dict[str, np.ndarray] = {}
for func_type, df_g in nsd.groupby("func_type"):
    ratio = _ratio_curve_for_group(df_g)
    if ratio is None:
        continue
    curves_by_type[str(func_type)] = ratio

if not curves_by_size:
    raise ValueError("No NSD groups produced warm data for `size`.")
if not curves_by_type:
    raise ValueError("No NSD groups produced warm data for function types derived from `func_name`.")

# --- Plot (two subplots) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.66, 3.2), dpi=300, sharey=True)

# Left: by size
for size in sorted(
    curves_by_size.keys(),
    key=lambda x: float(x) if str(x).replace(".", "", 1).isdigit() else x,
):
    ax1.plot(PERCENTILE_GRID, curves_by_size[size], label=str(size), alpha=0.9)
ax1.set_title("NSD: by size")
ax1.set_xlabel("Percentile (%)")
ax1.set_ylabel("Cold Factor (CF)")
ax1.set_xlim(80, 100)
ax1.set_yscale("log", base=10)
ax1.set_ylim(bottom=1, top=100)
ax1.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:g}"))
ax1.minorticks_off()
ax1.grid(True, linestyle="--", alpha=0.3, linewidth=0.6)
ax1.legend(frameon=False, fontsize=6, loc="upper left")

# Right: by function type
for func_type in sorted(curves_by_type.keys()):
    ax2.plot(PERCENTILE_GRID, curves_by_type[func_type], label=str(func_type), alpha=0.9)
ax2.set_title("NSD: by func type")
ax2.set_xlabel("Percentile (%)")
ax2.set_xlim(80, 100)
ax2.set_yscale("log", base=10)
ax2.set_ylim(bottom=1, top=100)
ax2.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:g}"))
ax2.minorticks_off()
ax2.grid(True, linestyle="--", alpha=0.3, linewidth=0.6)
ax2.legend(frameon=False, fontsize=6, loc="upper left")

plt.tight_layout()
os.makedirs("../figures/tertiary", exist_ok=True)
plt.savefig("../figures/tertiary/nsd_ratio_curves_by_size_and_func_type.pdf", bbox_inches="tight")
plt.show()


<notebook cell>:142: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Intra-day percentiles by hour (SPM + NSD + Huawei in subplots)

from __future__ import annotations

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
from sqlalchemy import create_engine

# --- Config (paths relative to tertiary-repos/, where this notebook lives) ---
SPM_DB = Path("mahmoudi_et_al/experiments/results/results.duckdb")
SPM_TABLE = "results"
SPM_DURATION_COL = "client_elapsed_time"
SPM_ELAPSED_COL = "elapsed_time"
SPM_START_COL = "client_start_time"
SPM_IS_COLD_COL = "is_cold"

NSD_MYSQL_URL = "mysql+pymysql://root:@localhost/benchmark"
NSD_TABLE = "data"
NSD_DURATION_COL = "google_time"
NSD_INVOKE_COL = "invoke_time"
NSD_START_COL = "function_start"
NSD_STOP_COL = "function_stop"
NSD_ISCOLD_COL = "isCold"

# Path to the exported Huawei CSV
HUAWEI_CSV_PATH = Path("../data/tertiary_aggregations/reconstruct_virtual_hour_latency.csv")

# We need the median (0.50) to calculate the relative Tail Factor, plus the tails
PERCENTILES_CALC = [0.50, 0.95, 0.99, 0.999, 0.9999]
PERCENTILES_PLOT = {
    0.95: "p95", 
    0.99: "p99", 
    0.999: "p99.9", 
    0.9999: "p99.99"
}

# --- Paper Styling (ACM 1-Column) ---
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'lines.linewidth': 1.2,
})
perc_palette = dict(zip(PERCENTILES_PLOT.keys(), sns.color_palette("colorblind", n_colors=4)))

def _to_datetime_auto(series: pd.Series) -> pd.Series:
    """Automatically converts raw timestamps to datetimes by inferring the unit."""
    if pd.api.types.is_datetime64_any_dtype(series):
        return series
    numeric = pd.to_numeric(series, errors="coerce")
    if float(numeric.notna().mean()) >= 0.9:
        med = float(np.nanmedian(np.abs(numeric.to_numpy(dtype=float))))
        if med > 1e17: unit = "ns"
        elif med > 1e14: unit = "us"
        elif med > 1e11: unit = "ms"
        elif med > 1e8: unit = "s"
        else: unit = "ms"
        return pd.to_datetime(numeric, unit=unit, errors="coerce")
    return pd.to_datetime(series, errors="coerce")


# 1. SPM (DuckDB): client_elapsed_time - elapsed_time
if not SPM_DB.exists():
    print(f"Warning: SPM DuckDB not found: {SPM_DB.resolve()}")
    spm_stats = pd.DataFrame(columns=PERCENTILES_PLOT.keys(), index=range(24))
else:
    spm_df = duckdb.connect(str(SPM_DB)).execute(
        f"SELECT {SPM_DURATION_COL} AS client_elapsed_s, {SPM_ELAPSED_COL} AS elapsed_ms, {SPM_START_COL} AS start_time, {SPM_IS_COLD_COL} AS is_cold FROM {SPM_TABLE}"
    ).df()

    spm_df["client_elapsed_s"] = pd.to_numeric(spm_df["client_elapsed_s"], errors="coerce")
    spm_df["elapsed_ms"] = pd.to_numeric(spm_df["elapsed_ms"], errors="coerce")
    spm_df["latency_ms"] = spm_df["client_elapsed_s"] * 1000.0 - spm_df["elapsed_ms"]
    spm_df["start_time"] = pd.to_numeric(spm_df["start_time"], errors="coerce")

    is_cold_raw = spm_df["is_cold"].astype(str).str.strip().str.lower()
    warm_mask = is_cold_raw.isin(["false", "0", "no"])

    spm_df = spm_df.loc[warm_mask & np.isfinite(spm_df["latency_ms"]) & np.isfinite(spm_df["start_time"])].copy()
    if spm_df.empty:
        raise ValueError("SPM warm-only selection produced 0 rows.")

    spm_ts_vals = spm_df["start_time"].to_numpy()
    spm_is_ms = spm_ts_vals >= 1_000_000_000_000
    spm_ts_seconds = np.where(spm_is_ms, spm_ts_vals / 1000.0, spm_ts_vals)

    spm_df["timestamp"] = pd.to_datetime(spm_ts_seconds, unit="s", utc=True)
    spm_df["hour"] = spm_df["timestamp"].dt.hour

    spm_quant = spm_df.groupby("hour", dropna=False)["latency_ms"].quantile(PERCENTILES_CALC).unstack(level=-1)
    spm_stats = pd.DataFrame(index=range(24)).join(spm_quant)

    # Convert to relative Tail Factor (pXX / p50)
    for p in PERCENTILES_PLOT.keys():
        spm_stats[p] = spm_stats[p] / spm_stats[0.50]


# 2. NSD (MySQL): google_time - function runtime
try:
    engine = create_engine(NSD_MYSQL_URL)
    nsd_df = pd.read_sql(
        f"SELECT {NSD_DURATION_COL} AS google_time, {NSD_START_COL} AS function_start, {NSD_STOP_COL} AS function_stop, {NSD_INVOKE_COL} AS invoke_time, {NSD_ISCOLD_COL} AS is_cold FROM {NSD_TABLE}",
        engine,
    )

    # Filter out cold starts first to save computation
    is_cold_raw = nsd_df["is_cold"].astype(str).str.strip().str.lower()
    warm_mask = is_cold_raw == "false"
    nsd_df = nsd_df.loc[warm_mask].copy()

    # Calculate worker runtime
    start_dt = _to_datetime_auto(nsd_df["function_start"])
    stop_dt = _to_datetime_auto(nsd_df["function_stop"])
    runtime_ms = (stop_dt - start_dt).dt.total_seconds() * 1000.0

    # Auto-normalize google_time unit
    google_raw = pd.to_numeric(nsd_df["google_time"], errors="coerce")
    google_ms = google_raw.copy()
    
    try:
        runtime_med = float(np.nanmedian(runtime_ms.to_numpy(dtype=float)))
        google_med = float(np.nanmedian(google_raw.to_numpy(dtype=float)))
        if np.isfinite(runtime_med) and runtime_med > 0 and np.isfinite(google_med) and google_med > 0:
            if (google_med / runtime_med) < 0.05:
                google_ms = google_raw * 1000.0
    except Exception:
        pass

    # Isolate Platform Latency
    nsd_df["duration_ms"] = google_ms - runtime_ms
    nsd_df["invoke_time"] = pd.to_numeric(nsd_df["invoke_time"], errors="coerce")

    # Filter out invalid rows and negative latencies (clock drift)
    valid_mask = np.isfinite(nsd_df["duration_ms"]) & (nsd_df["duration_ms"] >= 0) & np.isfinite(nsd_df["invoke_time"])
    nsd_df = nsd_df.loc[valid_mask].copy()

    if nsd_df.empty:
        raise ValueError("NSD warm-only selection produced 0 rows.")

    nsd_df["invoke_dt"] = pd.to_datetime(nsd_df["invoke_time"], unit="s", utc=True)
    nsd_df["hour"] = nsd_df["invoke_dt"].dt.hour

    nsd_quant = nsd_df.groupby("hour", dropna=False)["duration_ms"].quantile(PERCENTILES_CALC).unstack(level=-1)
    nsd_stats = pd.DataFrame(index=range(24)).join(nsd_quant)

    # Convert to relative Tail Factor (pXX / p50)
    for p in PERCENTILES_PLOT.keys():
        nsd_stats[p] = nsd_stats[p] / nsd_stats[0.50]

except Exception as e:
    print(f"Warning: Could not load NSD data. Reason: {e}")
    nsd_stats = pd.DataFrame(columns=PERCENTILES_PLOT.keys(), index=range(24))


# 3. Huawei (CSV): Pre-calculated Relatives
if HUAWEI_CSV_PATH.exists():
    huawei_raw = pd.read_csv(HUAWEI_CSV_PATH)
    huawei_stats = huawei_raw.rename(columns={
        "virtual_hour": "hour",
        "rel_p95": 0.95,
        "rel_p99": 0.99,
        "rel_p999": 0.999,
        "rel_p9999": 0.9999
    }).set_index("hour")
    
    # Reindex to guarantee all 24 hours exist
    huawei_stats = huawei_stats.reindex(range(24))
    
    # Mathematical correction: The exported CSV contains relative increase (e.g. 0.5 for +50%). 
    # We add 1.0 to convert it to a Tail Factor (1.5) so it matches the SPM/NSD charts mathematically.
    huawei_stats = huawei_stats + 1.0
else:
    print(f"Warning: Huawei CSV not found at {HUAWEI_CSV_PATH.resolve()}")
    huawei_stats = pd.DataFrame(columns=PERCENTILES_PLOT.keys(), index=range(24))


fig, axes = plt.subplots(3, 1, figsize=(3.33, 4.5), sharex=True, dpi=300)

datasets = [
    (r"Lambda20$^{\alpha}$", spm_stats, 100), # 10^1 (SPM)
    ("GCF23", nsd_stats, 1000),                # 10^2 (NSD)
    ("Huawei25", huawei_stats, 100000)        # 10^5 (Huawei); peak TF is ~4.1e4
]

for ax, (title, df, ymax) in zip(axes, datasets):
    if df.dropna(how='all').empty:
        ax.text(0.5, 0.5, "Data Unavailable", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        continue
        
    for p, label in PERCENTILES_PLOT.items():
        if p in df.columns:
            ax.plot(
                df.index, 
                df[p], 
                marker="o", 
                markersize=2.2, 
                label=label, 
                color=perc_palette[p]
            )
            
    ax.set_title(title, pad=3)
    
    # Paper specifics: Log scale, explicit custom limits, clean grid
    ax.set_yscale("log", base=10)
    ax.set_ylim(1, ymax) # Baseline is always 1 (10^0), max varies per plot
    ax.minorticks_off()
    ax.grid(True, linestyle="--", alpha=0.3, linewidth=0.6)
    
    ax.set_xlim(0, 23)
    ax.set_xticks(range(0, 24, 4)) # Ticks every 4 hours to avoid crowding

# Global Labels
fig.supylabel("Tail Factor (TF)", fontsize=8)
axes[-1].set_xlabel("Hour of day", fontsize=8)

# Shared top legend
handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(
        handles, 
        labels, 
        loc="upper center", 
        bbox_to_anchor=(0.5, 1.05), 
        ncol=4, 
        frameon=False, 
        handlelength=1.2, 
        handletextpad=0.4, 
        columnspacing=1.0
    )

# Adjust spacing specifically to fit within 3.33in width without cutting off labels
plt.tight_layout(rect=[0.02, 0.0, 1, 0.95])
fig.savefig("../figures/tertiary/intraday_tail_factors_spm_nsd_huawei.pdf", bbox_inches="tight")
plt.show()

<notebook cell>:244: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

NSD_MYSQL_URL = "mysql+pymysql://root:@localhost/benchmark"
NSD_TABLE = "data"

PERCENTILES_CALC = [0.50, 0.95, 0.99, 0.999, 0.9999]

def _to_datetime_auto(series: pd.Series) -> pd.Series:
    """Automatically converts raw timestamps to datetimes by inferring the unit."""
    if pd.api.types.is_datetime64_any_dtype(series):
        return series
    numeric = pd.to_numeric(series, errors="coerce")
    if float(numeric.notna().mean()) >= 0.9:
        med = float(np.nanmedian(np.abs(numeric.to_numpy(dtype=float))))
        if med > 1e17: unit = "ns"
        elif med > 1e14: unit = "us"
        elif med > 1e11: unit = "ms"
        elif med > 1e8: unit = "s"
        else: unit = "ms"
        return pd.to_datetime(numeric, unit=unit, errors="coerce")
    return pd.to_datetime(series, errors="coerce")

def _as_warm_bool(series: pd.Series) -> pd.Series:
    """Normalize common boolean encodings and return a warm-mask."""
    s = series.astype(str).str.strip().str.lower()
    return s.isin(["false", "0", "no"])

print("Connecting to NSD Database...")
engine = create_engine(NSD_MYSQL_URL)
df = pd.read_sql(
    f"SELECT google_time, function_start, function_stop, invoke_time, type, isCold FROM {NSD_TABLE}",
    engine,
    )

# 2. Filter Warm Starts
warm_mask = pd.Series(False, index=df.index)
if "isCold" in df.columns:
    is_cold_raw = df["isCold"].astype(str).str.strip().str.lower()
    warm_mask = is_cold_raw == "false"
df = df.loc[warm_mask].copy()

# 3. Calculate Worker Runtime & Normalize Google Time
start_dt = _to_datetime_auto(df["function_start"])
stop_dt = _to_datetime_auto(df["function_stop"])
runtime_ms = (stop_dt - start_dt).dt.total_seconds() * 1000.0

google_raw = pd.to_numeric(df["google_time"], errors="coerce")
google_ms = google_raw.copy()

try:
    runtime_med = float(np.nanmedian(runtime_ms.to_numpy(dtype=float)))
    google_med = float(np.nanmedian(google_raw.to_numpy(dtype=float)))
    if np.isfinite(runtime_med) and runtime_med > 0 and np.isfinite(google_med) and google_med > 0:
        if (google_med / runtime_med) < 0.05:
            google_ms = google_raw * 1000.0
except Exception:
    pass

# 4. Isolate Platform Latency and Hour
df["duration_ms"] = google_ms - runtime_ms
df["invoke_time"] = pd.to_numeric(df["invoke_time"], errors="coerce")

# Filter valid rows and negative clock drift
df = df.dropna(subset=["duration_ms", "invoke_time"])
df = df[df["duration_ms"] >= 0].copy()

if df.empty:
    raise ValueError("NSD warm selection produced 0 valid rows.")

df["hour"] = pd.to_datetime(df["invoke_time"], unit="s", utc=True).dt.hour

# 5. Group by Hour and Calculate Quantiles
nsd_quant = df.groupby("hour", dropna=False)["duration_ms"].quantile(PERCENTILES_CALC).unstack(level=-1)
# Ensure all 24 hours exist in the dataframe
nsd_quant = pd.DataFrame(index=range(24)).join(nsd_quant)

# 6. Plotting
plt.figure(figsize=(10, 5), dpi=150)

plt.plot(nsd_quant.index, nsd_quant[0.50], label='Median (p50)', color='black', linewidth=2.5, linestyle='--')

plt.plot(nsd_quant.index, nsd_quant[0.95], label='p95', marker='o', markersize=4)
plt.plot(nsd_quant.index, nsd_quant[0.99], label='p99', marker='s', markersize=4)
plt.plot(nsd_quant.index, nsd_quant[0.999], label='p99.9', marker='^', markersize=4)
plt.plot(nsd_quant.index, nsd_quant[0.9999], label='p99.99', marker='d', markersize=4)

plt.title('Absolute Platform Latency by Hour (NSD / GCF Warm Starts)')
plt.xlabel('Hour of Day (0-23 UTC)')
plt.ylabel('Absolute Latency (ms)')
plt.xticks(range(0, 24))

# Log scale to handle the massive variance between median and tail
plt.yscale('log', base=10)

plt.grid(True, which="both", linestyle='--', alpha=0.4)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Percentiles")
plt.tight_layout()

plt.show()

Connecting to NSD Database...


<notebook cell>:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Configuration Tail-Factor Heatmap
from __future__ import annotations

import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# --- 1. Config & Load Data ---
# Path to the exported CSV
CSV_PATH = Path("../data/tertiary_aggregations/warm_platform_latency_tail_factors.csv")

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Data file not found at: {CSV_PATH.resolve()}")

df = pd.read_csv(CSV_PATH)

# Configurations backed by too few invocations produce tail factors that are an artefact of the
# sample size rather than a property of the platform (the smallest cells hold 19-290 calls, so
# "p99.99" there is a single observation). Use the same floor as hua_warm_share_by_config.csv,
# which applies HAVING count(*) >= 1000 in SQL.
MIN_CALLS = 1000
dropped = df.loc[df["n"] < MIN_CALLS, ["config", "n"]]
if not dropped.empty:
    print(f"Excluding {len(dropped)} configuration(s) with n < {MIN_CALLS:,}:")
    for cfg, n in dropped.itertuples(index=False):
        print(f"  {cfg:<20} n = {n:,}")
df = df[df["n"] >= MIN_CALLS].reset_index(drop=True)
print(f"Plotting {len(df)} configurations (n = {df['n'].min():,} .. {df['n'].max():,})")

# --- 2. Clean & Sort Data ---
# Extract MB integer for logical sorting (smallest to largest config)
def extract_mb(config_str: str) -> int:
    match = re.search(r'(\d+)MB', str(config_str))
    return int(match.group(1)) if match else 0

df['sort_key'] = df['config'].apply(extract_mb)
df = df.sort_values('sort_key').reset_index(drop=True)

row_labels = df['config'].tolist()
PERCENTILES = ["95", "99", "99.9", "99.99"]
col_labels = [fr"$TF_{{p{p}}}$" for p in PERCENTILES]
matrix_cols = ["p95_ratio", "p99_ratio", "p99.9_ratio", "p99.99_ratio"]

# Ensure columns exist in the loaded CSV
missing_cols = [c for c in matrix_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns in CSV: {missing_cols}")

matrix = df[matrix_cols].to_numpy()

# --- 3. Paper-Friendly Formatting ---
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
})

# Standard 1-column ACM width is 3.33in. 
fig_w = 3.33
fig_h = max(2.5, 0.22 * len(row_labels))
fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=300)

# Set up Logarithmic scale
vmin = float(np.nanmin(matrix[matrix > 0]))
vmax = float(np.nanmax(matrix))
norm = LogNorm(vmin=vmin, vmax=vmax)

im = ax.imshow(matrix, aspect="auto", cmap="viridis", norm=norm)

# Colorbar configuration
cbar = fig.colorbar(im, ax=ax, pad=0.03, fraction=0.06)
cbar.set_label("Tail Factor (TF)", fontsize=8)
cbar.ax.tick_params(labelsize=7)
cbar.ax.minorticks_off()

# Axes Ticks & Labels
ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=7)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=7)

ax.tick_params(axis="y", pad=4)
ax.tick_params(axis="x", pad=4)
ax.set_xlabel("Percentile", fontsize=8)

# Annotate values with dynamic contrast AND dynamic formatting
for i in range(len(row_labels)):
    for j in range(len(col_labels)):
        v = matrix[i, j]
        if np.isfinite(v):
            # Contrast logic: Viridis turns bright > ~0.55 in LogNorm space
            norm_val = norm(v)
            text_color = "black" if norm_val > 0.55 else "white"
            
            # Format logic: Keep decimals for small numbers, drop them for huge numbers to save space
            if v < 100:
                txt = f"{v:.1f}"
            else:
                txt = f"{int(round(v))}"
                
            ax.text(j, i, txt, ha="center", va="center", color=text_color, fontsize=7)

# Adjust margins to accommodate the lengthy config labels on the left
fig.subplots_adjust(left=0.45, right=0.90, bottom=0.15, top=0.95)

plt.savefig("../figures/tertiary/config_tail_factor_heatmap.pdf", bbox_inches="tight")
plt.show()

Excluding 5 configuration(s) with n < 1,000:
  3000mc / 3584MB      n = 290
  1600mc / 1792MB      n = 279
  2200mc / 2560MB      n = 260
  1200mc / 1280MB      n = 250
  8200mc / 10240MB     n = 19
Plotting 7 configurations (n = 12,470 .. 301,249,341)


<notebook cell>:114: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
